In [1]:
import os
import numpy as np
import matplotlib.pyplot as pl

In [2]:
os.chdir('C:\cygwin64\home\codes\dispatch2\data\eos\Tomida+Hori_2016')
import EOS

In [3]:
eos=EOS.eos_t()

reading data from table tomida.dat


In [4]:
d=1e-12
T=100
P=eos.pressure(T,d)
U=eos.U(T,d)
P,U

(0.0035106052664206065, 8461278696.0848522)

In [5]:
pl.figure(1)
pl.clf()
T=np.logspace(2.,3.5)
for d in (1e-14,1e-13,1e-12,1e-11):
    U=eos.U(T,d)
    pl.loglog(T,U)
pl.xlabel('T [K]')
pl.ylabel('U [erg/g]');

To follow an adiabat we need to have $$\Delta U = \frac{P}{\rho}\Delta\ln\rho,$$ where 

$$\Delta U=U(T+\Delta T,\ln\rho+\Delta\ln\rho)-U(T,\ln\rho),$$ 

and since $U$ depends mainly on $T$, this may be approximated by

$$U \frac{\partial \ln U}{\partial \ln T} \Delta\ln T \approx  \frac{P}{\rho}\Delta\ln\rho,$$
or 
$$\frac{\Delta\ln T }{\Delta\ln\rho} \approx  \frac{P}{\rho U\frac{\partial \ln U}{\partial \ln T}},$$


Compute adiabats, with 200 points covering 5 decades in density, starting from 9 different (rho,T) points:

Note that this step may be skipped if it has been executed before; then proceed to the restore!

In [67]:
adiabats={}
for T0 in [100]:
    for logd_min in [-12,-11,-10]:
        d=np.logspace(logd_min,logd_min+10,501)
        np.log10(d[1]/d[0])
        dlnd=np.log(d[1]/d[0])
        nd=len(d)
        T=np.zeros(nd)
        U=np.zeros(nd)
        P=np.zeros(nd)
        T[0]=T0
        U[0]=eos.U(T[0],d[0])
        P[0]=eos.pressure(T[0],d[0])

        for ir in range(1,nd):
            gamma=eos.gamma(T[ir-1],d[ir-1])
            T[ir]=T[ir-1]*np.exp(dlnd*(gamma-1.0))
            dlnT=1.0
            nit=0
            while(abs(dlnT) > 1e-4):
                U[ir]=eos.U(T[ir],d[ir])
                P[ir]=eos.pressure(T[ir],d[ir])
                d1=(d[ir]+d[ir-1])/2.0
                T1=(T[ir]+T[ir-1])/2.0
                U1=(U[ir]+U[ir-1])/2.0
                P1=(P[ir]+P[ir-1])/2.0
                dUdlnT=U1*eos.dlnUdlnT(T1,d1)
                dU=U[ir]-U[ir-1]
                Pd=(P[ir]+P[ir-1])/(d[ir]+d[ir-1])
                Eps=dU-Pd*dlnd
                dEpsdlnT=dUdlnT-P1/d1*dlnd
                dlnT=-Eps/dEpsdlnT
                T[ir]=T[ir]*np.exp(dlnT)
                nit+=1
        ir=nd-1
        key='T{:2d}d-{:02d}'.format(T0,-logd_min)
        print('{} {:2d}  T:{:7.1f}  P:{:11.3e}  nit:{:2d}'.format(
            key,ir,T[ir],P[ir],nit))
        adiabats[key]={'T':T,'P':P,'U':U,'d':d}

T100d-12 500  T:35640.7  P:  2.661e+10  nit: 2
T100d-11 500  T:66971.0  P:  5.152e+11  nit: 2
T100d-10 500  T:141885.9  P:  9.644e+12  nit: 2


In [ ]:
import save_restore as sr
sr.save('Jupiter.save',adiabats)

Restore from this point when restarting

In [18]:
import save_restore as sr
adiabats=sr.restore('Jupiter.save')
adiabats.keys()

dict_keys(['T100d-12', 'T100d-11', 'T100d-10'])

In [30]:
pl.figure(1); pl.clf(); pl.title('adiabats')
pl.xlabel('d [cgs]'); pl.ylabel('T [K]')
for k,v in adiabats.items():
  if k[0:3]=='T10':
    if not type(v)==str:
        pl.loglog(v['d'],v['T'])
pl.savefig('adiabat_fig1.png')
pl.figure(2); pl.clf(); pl.title('adiabats')
pl.xlabel('d [cgs]'); pl.ylabel('P [bar]')
for k,v in adiabats.items():
  if k[0:3]=='T10':
    if not type(v)==str:
        pl.loglog(v['d'],1e-6*v['P'])
pl.savefig('adiabat_fig2.png')
pl.figure(3); pl.clf(); pl.title('adiabats')
pl.xlabel('d [cgs]'); pl.ylabel('U [erg/g]')
for k,v in adiabats.items():
  if k[0:3]=='T10':
    if not type(v)==str:
        pl.loglog(v['d'],v['U'])
pl.savefig('adiabat_fig3.png')

Now that we have an adiabat, we can integrate the hydrostatic pressure balance equation, turning $\Delta\ln P$ into $\Delta \ln r$, where $r$ is radius:

$$\frac{dP}{dR}=-g\rho,$$
so
$$\frac{d\ln P}{d\ln R}=-\frac{g\rho R}{P}=-\frac{G M\rho}{R P},$$

$$\frac{d\ln R}{d\ln P}=-\frac{P}{g\rho R}=-\frac{R P}{G M\rho},$$


In [17]:
from scaling import cgs
for i in (4,5,6):
    pl.figure(i); pl.clf()
Me=[1099,598,355.6]
Me=[1099]
j=0
for k,v in adiabats.items():
    if k[0:7]=='T100d-1':
        me=Me[j]
        j+=1
        RE=cgs.r_earth*me**0.333
        ME=cgs.m_earth*me
        P=v['P']
        d=v['d']
        T=v['T']
        nd=len(d)
        R=np.ones(nd)
        M=np.zeros(nd)
        R[0]=RE*245.
        i=0
        M[i]=ME
        while(i < nd-1 and R[i] > RE/50):
            i=i+1
            R[i]=R[i-1]
            for nit in range(5):
                F=(R[i]*P[i]/d[i]+R[i-1]*P[i-1]/d[i-1])/2.0
                dlnRdlnP=F/(cgs.grav*ME)
                dlnP=np.log(P[i-1]/P[i])
                dlnR=dlnP*dlnRdlnP
                R[i]=R[i-1]*np.exp(dlnR)
            dME=4./3.*np.pi*(R[i-1]**3-R[i]**3)*(d[i]+d[i-1])/2.
            ME-=dME
            M[i]=ME
            print('    {:2d}  dlnRdlnP:{:9.3f}  R:{:9.3f}  M:{:9.3f}'.format(
                i-1,dlnRdlnP,R[i-1]/cgs.r_earth,M[i-1]/cgs.m_earth))
        #print('{:3d}{:14.3e}{:14.3e}{:14.3e}{:14.3e}{:14.3e}'.format(
        #      i-1,dlnRdlnP,d[i-1],T[i-1],R[i-1]/cgs.r_earth),
        #      M[i-1]/cgs.m_earth)
        print(i-1,d[i-1])
        title='Embryo mass: 350, 595, 1095'.format(me)
        pl.figure(4)
        pl.xlabel('R/Re'); pl.ylabel('P [bar]')
        pl.title(title)
        pl.loglog(R[0:i]/cgs.r_earth,1e-6*P[0:i],'-+')
        pl.savefig('Jupiter_fig4.png')
        pl.figure(5)
        pl.xlabel('R/Re'); pl.ylabel('T [K]')
        pl.title(title)
        pl.loglog(R[0:i]/cgs.r_earth,T[0:i])
        pl.savefig('Jupiter_fig5.png')
        pl.figure(6)
        pl.xlabel('R/Re'); pl.ylabel('M [M_E]')
        pl.title(title)
        pl.loglog(R[0:i]/cgs.r_earth,M[0:i]/cgs.m_earth,'-+')
        pl.savefig('Jupiter_fig6.png')

     0  dlnRdlnP:    0.013  R: 2522.425  M: 1099.000
     1  dlnRdlnP:    0.013  R: 2520.233  M: 1099.000
     2  dlnRdlnP:    0.014  R: 2517.992  M: 1099.000
     3  dlnRdlnP:    0.014  R: 2515.705  M: 1099.000
     4  dlnRdlnP:    0.014  R: 2513.365  M: 1099.000
     5  dlnRdlnP:    0.014  R: 2510.976  M: 1099.000
     6  dlnRdlnP:    0.015  R: 2508.531  M: 1099.000
     7  dlnRdlnP:    0.015  R: 2506.037  M: 1099.000
     8  dlnRdlnP:    0.015  R: 2503.484  M: 1099.000
     9  dlnRdlnP:    0.016  R: 2500.881  M: 1099.000
    10  dlnRdlnP:    0.016  R: 2498.216  M: 1099.000
    11  dlnRdlnP:    0.016  R: 2495.500  M: 1099.000
    12  dlnRdlnP:    0.017  R: 2492.721  M: 1099.000
    13  dlnRdlnP:    0.017  R: 2489.889  M: 1099.000
    14  dlnRdlnP:    0.017  R: 2486.992  M: 1099.000
    15  dlnRdlnP:    0.018  R: 2484.042  M: 1099.000
    16  dlnRdlnP:    0.018  R: 2481.025  M: 1099.000
    17  dlnRdlnP:    0.019  R: 2477.954  M: 1099.000
    18  dlnRdlnP:    0.019  R: 2474.814  M: 10

IndexError: list index out of range